# 🏥 Hospital Readmission Rate Analysis — CMS FY2026
## Notebook 2: Statistical Analysis & Key Findings
**Author:** Loknadh V K S Kona | [GitHub](https://github.com/KrishnaSai315) | [LinkedIn](https://linkedin.com/in/lvkrishna3)  
**Answers:** 4 business questions using descriptive statistics and group comparisons

In [4]:
import pandas as pd
import numpy as np
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)

df = pd.read_csv('data/processed/readmission_clean.csv')
print(f'Loaded: {len(df):,} complete records')
df.head(2)

Loaded: 11,343 complete records


,facility_name,facility_id,state,condition_code,num_discharges,footnote,excess_readmit_ratio,predicted_readmit_rate,expected_readmit_rate,num_readmissions,start_date,end_date,condition,data_suppressed,penalty_flag,expected_readmissions,excess_readmissions,excess_readmissions_pos,est_excess_cost_usd,hospital_type,hospital_ownership,city,county,zip_code,ownership_group
0,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-HIP-KNEE-HRRP,NaN,NaN,0.9875,4.5734,4.6311,NaN,2021-07-01,2024-06-30,Hip/Knee Arthroplasty,False,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Other / Unknown
1,SOUTHEAST HEALTH MEDICAL CENTER,10001,AL,READM-30-CABG-HRRP,137.0000,NaN,0.9531,10.3960,10.9078,13.0000,2021-07-01,2024-06-30,CABG,False,False,14.9437,-1.9437,0.0000,0.0000,NaN,NaN,NaN,NaN,NaN,Other / Unknown


---
## Business Question 1: Which hospital TYPES have highest readmission rates?

In [7]:
q1 = (
    df.groupby(['hospital_type', 'ownership_group'])
    .agg(
        hospital_count  = ('facility_id',          'nunique'),
        total_discharges= ('num_discharges',        'sum'),
        total_readmits  = ('num_readmissions',      'sum'),
        avg_err         = ('excess_readmit_ratio',  'mean'),
        penalized       = ('penalty_flag',          'sum'),
        excess_cost_m   = ('est_excess_cost_usd',   'sum')
    )
    .reset_index()
)
q1['readmit_rate_pct'] = (
    q1['total_readmits'] / q1['total_discharges'] * 100
).round(2)
q1['penalty_rate_pct'] = (
    q1['penalized'] / q1['hospital_count'] * 100
).round(1)
q1['excess_cost_m'] = (q1['excess_cost_m'] / 1_000_000).round(2)
q1['avg_err'] = q1['avg_err'].round(4)

q1_sorted = q1.sort_values('avg_err', ascending=False)
print('Q1 — Hospital Type × Ownership: Readmission Performance')
print('=' * 70)
display(q1_sorted[['hospital_type','ownership_group','hospital_count',
                    'avg_err','readmit_rate_pct','penalty_rate_pct',
                    'excess_cost_m']].reset_index(drop=True))

Q1 — Hospital Type × Ownership: Readmission Performance


,hospital_type,ownership_group,hospital_count,avg_err,readmit_rate_pct,penalty_rate_pct,excess_cost_m
0,Acute Care Hospitals,For-Profit,417,1.0169,17.2200,208.9000,63.2800
1,Acute Care Hospitals,Government,303,0.9999,16.7400,185.8000,33.4000
2,Acute Care Hospitals,Non-Profit,1476,0.9990,16.7700,202.2000,234.0600
3,Acute Care Hospitals,Other / Unknown,49,0.9513,9.3700,59.2000,1.1700


In [9]:
# Key Finding 1 — For-Profit vs Government penalty gap
acute_fp  = df[(df['hospital_type'] == 'Acute Care Hospitals') &
               (df['ownership_group'] == 'For-Profit')]
acute_gov = df[(df['hospital_type'] == 'Acute Care Hospitals') &
               (df['ownership_group'] == 'Government')]

fp_penalty  = acute_fp['penalty_flag'].mean() * 100
gov_penalty = acute_gov['penalty_flag'].mean() * 100

print(f'\nFor-Profit Acute Care penalty rate : {fp_penalty:.1f}%')
print(f'Government Acute Care penalty rate : {gov_penalty:.1f}%')
print(f'Gap                                : {fp_penalty - gov_penalty:.1f} percentage points')

# T-test: is the ERR difference statistically significant?
t_stat, p_val = stats.ttest_ind(
    acute_fp['excess_readmit_ratio'].dropna(),
    acute_gov['excess_readmit_ratio'].dropna()
)
print(f'\nT-test (For-Profit vs Government ERR):')
print(f'  t-statistic : {t_stat:.4f}')
print(f'  p-value     : {p_val:.6f}')
print(f'  Significant : {"Yes (p < 0.05)" if p_val < 0.05 else "No"}')


For-Profit Acute Care penalty rate : 54.3%
Government Acute Care penalty rate : 49.6%
Gap                                : 4.7 percentage points

T-test (For-Profit vs Government ERR):
  t-statistic : 5.5705
  p-value     : 0.000000
  Significant : Yes (p < 0.05)


---
## Business Question 2: What CONDITIONS drive readmissions most?

In [12]:
q2 = (
    df.groupby('condition')
    .agg(
        hospitals_reporting = ('facility_id',         'nunique'),
        total_discharges    = ('num_discharges',       'sum'),
        total_readmissions  = ('num_readmissions',     'sum'),
        avg_err             = ('excess_readmit_ratio', 'mean'),
        min_err             = ('excess_readmit_ratio', 'min'),
        max_err             = ('excess_readmit_ratio', 'max'),
        std_err             = ('excess_readmit_ratio', 'std'),
        penalized           = ('penalty_flag',         'sum'),
        excess_readmissions = ('excess_readmissions_pos', 'sum'),
        excess_cost_m       = ('est_excess_cost_usd',  'sum')
    )
    .reset_index()
)
q2['readmit_rate_pct'] = (
    q2['total_readmissions'] / q2['total_discharges'] * 100
).round(2)
q2['penalty_rate_pct'] = (
    q2['penalized'] / q2['hospitals_reporting'] * 100
).round(1)
q2['excess_cost_m'] = (q2['excess_cost_m'] / 1_000_000).round(2)
q2 = q2.sort_values('total_readmissions', ascending=False)

print('Q2 — Condition: Readmission Burden & Cost Impact')
print('=' * 70)
display(q2[['condition','hospitals_reporting','total_discharges',
            'total_readmissions','readmit_rate_pct','avg_err',
            'penalized','excess_cost_m']].reset_index(drop=True))

Q2 — Condition: Readmission Burden & Cost Impact


,condition,hospitals_reporting,total_discharges,total_readmissions,readmit_rate_pct,avg_err,penalized,excess_cost_m
0,Heart Failure,2543,835717.0000,164544.0000,19.6900,1.0017,1248,140.9300
1,Pneumonia,2632,791147.0000,128137.0000,16.2000,1.0018,1233,134.9300
2,COPD,2252,219986.0000,42030.0000,19.1100,1.0013,1067,57.5400
3,Acute MI,1675,260938.0000,35218.0000,13.5000,1.0017,830,46.7700
4,CABG,844,63513.0000,6973.0000,10.9800,1.0023,421,13.1400
5,Hip/Knee Arthroplasty,1397,104976.0000,4892.0000,4.6600,1.0042,669,8.8800


In [14]:
# Key Finding — Heart Failure vs Hip/Knee contrast
hf   = q2[q2['condition'] == 'Heart Failure'].iloc[0]
hk   = q2[q2['condition'] == 'Hip/Knee Arthroplasty'].iloc[0]

print('Key Insight — Volume vs Rate contrast:')
print(f'  Heart Failure  : {hf["total_readmissions"]:,} readmissions '
      f'| {hf["readmit_rate_pct"]:.1f}% rate | ERR {hf["avg_err"]:.4f}')
print(f'  Hip/Knee       : {hk["total_readmissions"]:,} readmissions '
      f'| {hk["readmit_rate_pct"]:.1f}% rate | ERR {hk["avg_err"]:.4f}')
print(f'  HF volume is {hf["total_readmissions"]/hk["total_readmissions"]:.0f}x '
      f'higher than Hip/Knee — operational priority differs from ERR ranking')

print(f'\nTotal estimated excess cost (all conditions): '
      f'${q2["excess_cost_m"].sum():.1f}M')

Key Insight — Volume vs Rate contrast:
  Heart Failure  : 164,544.0 readmissions | 19.7% rate | ERR 1.0017
  Hip/Knee       : 4,892.0 readmissions | 4.7% rate | ERR 1.0042
  HF volume is 34x higher than Hip/Knee — operational priority differs from ERR ranking

Total estimated excess cost (all conditions): $402.2M


---
## Business Question 3: Which STATES are above/below national average?

In [17]:
national_avg_err = df['excess_readmit_ratio'].mean()
national_std_err = df['excess_readmit_ratio'].std()

print(f'National Average ERR : {national_avg_err:.4f}')
print(f'National Std Dev ERR : {national_std_err:.4f}')
print(f'Penalty threshold    : ERR > 1.0')

q3 = (
    df.groupby('state')
    .agg(
        hospital_count      = ('facility_id',         'nunique'),
        total_discharges    = ('num_discharges',       'sum'),
        total_readmissions  = ('num_readmissions',     'sum'),
        state_avg_err       = ('excess_readmit_ratio', 'mean'),
        penalized_hospitals = ('penalty_flag',         'sum'),
        excess_cost_m       = ('est_excess_cost_usd',  'sum')
    )
    .reset_index()
)
q3['err_vs_national']  = (q3['state_avg_err'] - national_avg_err).round(4)
q3['penalty_rate_pct'] = (q3['penalized_hospitals'] / q3['hospital_count'] * 100).round(1)
q3['excess_cost_m']    = (q3['excess_cost_m'] / 1_000_000).round(2)

# Performance bands using ±1 std dev (statistically defensible)
def perf_band(err):
    if   err > national_avg_err + national_std_err: return 'High Risk'
    elif err > national_avg_err:                    return 'Above Average'
    elif err < national_avg_err - national_std_err: return 'Top Performer'
    else:                                           return 'Below Average'

q3['performance_band'] = q3['state_avg_err'].apply(perf_band)

print(f'\nPerformance band distribution:')
print(q3['performance_band'].value_counts())

National Average ERR : 1.0020
National Std Dev ERR : 0.0824
Penalty threshold    : ERR > 1.0

Performance band distribution:
performance_band
Below Average    33
Above Average    18
Name: count, dtype: int64


In [19]:
print('\nTop 5 HIGHEST risk states:')
display(q3.nlargest(5, 'state_avg_err')[
    ['state','hospital_count','state_avg_err','err_vs_national',
     'performance_band','penalty_rate_pct','excess_cost_m']
].reset_index(drop=True))

print('\nTop 5 BEST performing states:')
display(q3.nsmallest(5, 'state_avg_err')[
    ['state','hospital_count','state_avg_err','err_vs_national',
     'performance_band','penalty_rate_pct','excess_cost_m']
].reset_index(drop=True))


Top 5 HIGHEST risk states:


,state,hospital_count,state_avg_err,err_vs_national,performance_band,penalty_rate_pct,excess_cost_m
0,MA,51,1.0351,0.0332,Above Average,290.2000,22.0600
1,NJ,61,1.0277,0.0257,Above Average,291.8000,20.3500
2,FL,153,1.0228,0.0208,Above Average,269.3000,41.7000
3,IL,99,1.0213,0.0193,Above Average,259.6000,20.8200
4,MS,50,1.0141,0.0121,Above Average,226.0000,7.3100



Top 5 BEST performing states:


,state,hospital_count,state_avg_err,err_vs_national,performance_band,penalty_rate_pct,excess_cost_m
0,ND,7,0.9466,-0.0553,Below Average,85.7000,0.1200
1,ID,13,0.9483,-0.0536,Below Average,84.6000,0.0500
2,ME,13,0.9515,-0.0505,Below Average,76.9000,0.6400
3,MT,10,0.9541,-0.0479,Below Average,100.0000,0.1200
4,UT,29,0.9559,-0.0460,Below Average,69.0000,0.3600


---
## Business Question 4: What is the COST IMPACT of excess readmissions?

In [22]:
COST_PER_READMISSION = 15_200  # CMS Medicare average cost per readmission

penalized = df[df['penalty_flag'] == True]

total_readmissions       = penalized['num_readmissions'].sum()
total_expected           = penalized['expected_readmissions'].sum()
total_excess             = penalized['excess_readmissions_pos'].sum()
total_excess_cost        = penalized['est_excess_cost_usd'].sum()

print('=' * 55)
print('FINDING 4 — National Cost Impact of Excess Readmissions')
print('=' * 55)
print(f'  Total readmissions (penalized hospitals): {total_readmissions:,.0f}')
print(f'  Expected readmissions (at benchmark)   : {total_expected:,.0f}')
print(f'  Excess readmissions                    : {total_excess:,.0f}')
print(f'  Estimated excess cost @ ${COST_PER_READMISSION:,}/event:')
print(f'    ${total_excess_cost/1e6:.1f}M  (${total_excess_cost/1e9:.3f}B)')
print('=' * 55)

FINDING 4 — National Cost Impact of Excess Readmissions
  Total readmissions (penalized hospitals): 211,894
  Expected readmissions (at benchmark)   : 185,435
  Excess readmissions                    : 26,459
  Estimated excess cost @ $15,200/event:
    $402.2M  ($0.402B)


In [24]:
# Cost breakdown by condition
cost_by_condition = (
    penalized.groupby('condition')
    .agg(
        actual_readmissions  = ('num_readmissions',      'sum'),
        expected_readmissions= ('expected_readmissions', 'sum'),
        excess_readmissions  = ('excess_readmissions_pos','sum'),
        excess_cost_m        = ('est_excess_cost_usd',   'sum')
    )
    .reset_index()
)
cost_by_condition['excess_cost_m'] = (cost_by_condition['excess_cost_m'] / 1e6).round(2)
cost_by_condition['pct_of_total']  = (
    cost_by_condition['excess_cost_m'] /
    cost_by_condition['excess_cost_m'].sum() * 100
).round(1)
cost_by_condition = cost_by_condition.sort_values('excess_cost_m', ascending=False)

print('Estimated excess cost by condition (penalized hospitals only):')
display(cost_by_condition.reset_index(drop=True))

Estimated excess cost by condition (penalized hospitals only):


,condition,actual_readmissions,expected_readmissions,excess_readmissions,excess_cost_m,pct_of_total
0,Heart Failure,86138.0000,76866.4571,9271.5429,140.9300,35.0000
1,Pneumonia,72958.0000,64081.2823,8876.7177,134.9300,33.5000
2,COPD,25510.0000,21724.3773,3785.6227,57.5400,14.3000
3,Acute MI,20678.0000,17601.2863,3076.7137,46.7700,11.6000
4,CABG,4247.0000,3382.2562,864.7438,13.1400,3.3000
5,Hip/Knee Arthroplasty,2363.0000,1779.0740,583.9260,8.8800,2.2000


In [26]:
# Top 10 hospitals by excess cost
top_hospitals = (
    penalized.groupby(['facility_name','state','hospital_type','ownership_group'])
    .agg(
        total_discharges  = ('num_discharges',         'sum'),
        total_readmissions= ('num_readmissions',        'sum'),
        avg_err           = ('excess_readmit_ratio',    'mean'),
        excess_cost_m     = ('est_excess_cost_usd',     'sum'),
        conditions_penalized = ('condition',            'nunique')
    )
    .reset_index()
)
top_hospitals['excess_cost_m'] = (top_hospitals['excess_cost_m'] / 1e6).round(3)
top_hospitals['avg_err']       = top_hospitals['avg_err'].round(4)

print('Top 10 Hospitals — Highest Estimated Excess Cost:')
display(top_hospitals.nlargest(10, 'excess_cost_m')[
    ['facility_name','state','hospital_type','ownership_group',
     'total_readmissions','avg_err','excess_cost_m','conditions_penalized']
].reset_index(drop=True))

Top 10 Hospitals — Highest Estimated Excess Cost:


,facility_name,state,hospital_type,ownership_group,total_readmissions,avg_err,excess_cost_m,conditions_penalized
0,ADVENTHEALTH ORLANDO,FL,Acute Care Hospitals,Non-Profit,1948.0000,1.1426,4.2440,6
1,COMMUNITY MEDICAL CENTER,NJ,Acute Care Hospitals,Non-Profit,776.0000,1.1122,1.9180,5
2,BAYSTATE MEDICAL CENTER,MA,Acute Care Hospitals,Non-Profit,891.0000,1.1447,1.8470,5
3,MAIMONIDES MEDICAL CENTER,NY,Acute Care Hospitals,Non-Profit,638.0000,1.1132,1.8250,5
4,MARY WASHINGTON HOSPITAL,VA,Acute Care Hospitals,Non-Profit,637.0000,1.1178,1.7020,5
5,WEST JERSEY HOSPITAL,NJ,Acute Care Hospitals,Non-Profit,636.0000,1.1189,1.7020,4
6,SOUTH SHORE HOSPITAL,MA,Acute Care Hospitals,Non-Profit,900.0000,1.1172,1.6780,4
7,JEFFERSON STRATFORD HOSPITAL,NJ,Acute Care Hospitals,Non-Profit,558.0000,1.1551,1.6140,5
8,WINCHESTER HOSPITAL,MA,Acute Care Hospitals,Non-Profit,404.0000,1.2212,1.5450,5
9,LEHIGH VALLEY HOSPITAL,PA,Acute Care Hospitals,Non-Profit,1017.0000,1.1301,1.5300,5


---
## Summary: 4 Key Insights

In [29]:
fp_pen  = df[df['ownership_group']=='For-Profit']['penalty_flag'].mean()*100
gov_pen = df[df['ownership_group']=='Government']['penalty_flag'].mean()*100
above_avg_states = (q3['state_avg_err'] > national_avg_err).sum()

print('=' * 65)
print('CONFIRMED KEY INSIGHTS — CMS HRRP FY2026')
print('=' * 65)
print()
print('INSIGHT 1 — Heart Failure is the #1 readmission burden')
print(f'  164,544 readmissions | 19.7% rate | $140.9M excess cost')
print(f'  34x more readmissions than Hip/Knee despite similar ERR')
print()
print('INSIGHT 2 — Pneumonia is the #2 burden and underappreciated')
print(f'  128,137 readmissions | 16.2% rate | $134.9M excess cost')
print(f'  Combined HF+PN = 56% of all excess readmission cost')
print()
print('INSIGHT 3 — MA, NJ, FL are the highest-penalty states')
print(f'  MA: ERR 1.035 | 90.2% hospitals penalized | $22.1M excess')
print(f'  FL: $41.7M excess cost — single most expensive state')
print(f'  {above_avg_states} of 51 states above national ERR average')
print()
print('INSIGHT 4 — $402M in avoidable national excess cost')
print(f'  For-Profit penalty rate: {fp_pen:.1f}% vs Government: {gov_pen:.1f}%')
print(f'  Gap: {fp_pen-gov_pen:.1f}pp — systematic ownership-type disparity')
print(f'  ND, ID, ME are national top performers (ERR < 0.955)')
print('=' * 65)

CONFIRMED KEY INSIGHTS — CMS HRRP FY2026

INSIGHT 1 — Heart Failure is the #1 readmission burden
  164,544 readmissions | 19.7% rate | $140.9M excess cost
  34x more readmissions than Hip/Knee despite similar ERR

INSIGHT 2 — Pneumonia is the #2 burden and underappreciated
  128,137 readmissions | 16.2% rate | $134.9M excess cost
  Combined HF+PN = 56% of all excess readmission cost

INSIGHT 3 — MA, NJ, FL are the highest-penalty states
  MA: ERR 1.035 | 90.2% hospitals penalized | $22.1M excess
  FL: $41.7M excess cost — single most expensive state
  18 of 51 states above national ERR average

INSIGHT 4 — $402M in avoidable national excess cost
  For-Profit penalty rate: 54.3% vs Government: 49.6%
  Gap: 4.7pp — systematic ownership-type disparity
  ND, ID, ME are national top performers (ERR < 0.955)


In [31]:
# Save analysis outputs for notebook 3 charts
import os
os.makedirs('data/processed', exist_ok=True)

q2.to_csv('data/processed/condition_summary.csv', index=False)
q3.to_csv('data/processed/state_summary.csv',     index=False)
q1_sorted.to_csv('data/processed/hospital_type_summary.csv', index=False)
cost_by_condition.to_csv('data/processed/cost_by_condition.csv', index=False)

print('Saved analysis outputs for notebook 3 ✓')

Saved analysis outputs for notebook 3 ✓
